# Information Theory for ML: Entropy, KL Divergence, Cross-Entropy

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/information-theory)

We compute Shannon entropy and KL divergence from scratch, visualize the forward vs reverse KL mode-seeking vs mean-seeking behaviour, derive cross-entropy loss as NLL, and show how mutual information ranks features.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1 — Shannon entropy

In [ ]:
def entropy(p, base=2):
    """Shannon entropy of a discrete distribution. p should sum to 1."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0]   # 0 * log(0) = 0 by convention
    return -np.sum(p * np.log(p) / np.log(base))

# Entropy vs coin bias
bias = np.linspace(0.001, 0.999, 300)
H = [entropy([b, 1-b]) for b in bias]

plt.figure(figsize=(7, 4))
plt.plot(bias, H, color='#6366f1', lw=2)
plt.axvline(0.5, color='#f59e0b', ls='--', label='Max entropy at p=0.5 (H=1 bit)')
plt.xlabel('Coin bias p'); plt.ylabel('H(p) (bits)')
plt.title('Entropy of Bernoulli(p)'); plt.legend()
plt.tight_layout(); plt.show()

print(f'Fair coin (p=0.5):  H = {entropy([0.5, 0.5]):.4f} bits')
print(f'Biased  (p=0.9):    H = {entropy([0.9, 0.1]):.4f} bits')
print(f'Certain (p=1.0):    H = {entropy([1.0, 0.0]):.4f} bits')

## 2 — KL divergence: forward vs reverse

In [ ]:
def kl_divergence(p, q, eps=1e-10):
    """D_KL(P || Q) for discrete distributions."""
    p, q = np.asarray(p, float), np.asarray(q, float)
    p, q = p + eps, q + eps   # avoid log(0)
    p /= p.sum(); q /= q.sum()
    return np.sum(p * np.log(p / q))

# Bimodal true distribution P; compare forward vs reverse KL for a Gaussian Q
x = np.linspace(-6, 6, 500)
P_true = 0.5 * stats.norm.pdf(x, -2, 0.7) + 0.5 * stats.norm.pdf(x, 2, 0.7)  # bimodal
dx = x[1] - x[0]

# Forward KL (P||Q): Q must cover both modes → mean-seeking
# Reverse KL (Q||P): Q can collapse to one mode → mode-seeking
mu_range = np.linspace(-4, 4, 200)
sigma = 1.0  # fixed for demonstration

fwd_kl = []
rev_kl = []
for mu in mu_range:
    Q = stats.norm.pdf(x, mu, sigma)
    Q /= Q.sum() * dx
    P_norm = P_true / (P_true.sum() * dx)
    fwd_kl.append(np.sum(P_norm * np.log((P_norm + 1e-10) / (Q + 1e-10)) * dx))
    rev_kl.append(np.sum(Q * np.log((Q + 1e-10) / (P_norm + 1e-10)) * dx))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, kl_vals, title, color in zip(axes,
        [fwd_kl, rev_kl],
        ['Forward KL D(P||Q) — mean-seeking', 'Reverse KL D(Q||P) — mode-seeking'],
        ['#6366f1', '#f87171']):
    ax.plot(mu_range, kl_vals, color=color, lw=2)
    best_mu = mu_range[np.argmin(kl_vals)]
    ax.axvline(best_mu, ls='--', color='#f59e0b', label=f'Optimal μ = {best_mu:.2f}')
    ax.axvline(0, ls=':', color='gray', label='μ=0 (bimodal mean)')
    ax.set_xlabel('μ of Gaussian Q'); ax.set_ylabel('KL divergence'); ax.set_title(title); ax.legend()
plt.suptitle('Forward KL finds the mean between modes; Reverse KL commits to one mode')
plt.tight_layout(); plt.show()

## 3 — Cross-entropy = H(P) + KL(P||Q)

In [ ]:
# Verify H(P,Q) = H(P) + D_KL(P||Q) for discrete distributions
K = 5  # number of classes
P = np.array([0.4, 0.3, 0.15, 0.1, 0.05])  # true distribution
Q = np.array([0.25, 0.25, 0.2, 0.2, 0.1])  # model distribution

H_P = entropy(P, base=np.e)
KL_PQ = kl_divergence(P, Q)
HPQ_formula = H_P + KL_PQ
HPQ_direct = -np.sum(P * np.log(Q + 1e-10))

print(f'H(P)       = {H_P:.4f} nats')
print(f'KL(P||Q)   = {KL_PQ:.4f} nats')
print(f'H(P) + KL  = {HPQ_formula:.4f} nats')
print(f'H(P,Q) direct = {HPQ_direct:.4f} nats')
print(f'Equal? {abs(HPQ_formula - HPQ_direct) < 1e-8}')

# Minimum cross-entropy is H(P) — achieved when Q = P
print(f'\nMinimum possible H(P,Q) = H(P) = {H_P:.4f} (when Q=P)')

## 4 — Cross-entropy loss in PyTorch: NLL connection

In [ ]:
import torch
import torch.nn.functional as F

# Binary classification example
logits = torch.tensor([2.0, -1.0, 0.5, -0.3])
y_true = torch.tensor([1,    0,    1,   0   ], dtype=torch.long)

# Two ways to compute cross-entropy
y_pred_probs = torch.sigmoid(logits)
nll_manual = -torch.where(y_true == 1, torch.log(y_pred_probs), torch.log(1 - y_pred_probs)).mean()
bce_pytorch = F.binary_cross_entropy_with_logits(logits, y_true.float())

print(f'Manual NLL: {nll_manual.item():.6f}')
print(f'PyTorch BCE: {bce_pytorch.item():.6f}')
print(f'Equal: {abs(nll_manual.item() - bce_pytorch.item()) < 1e-5}')

## 5 — Mutual information for feature selection

In [ ]:
# Estimate I(X; Y) = H(Y) - H(Y|X) for discrete features and binary label
def mutual_information_discrete(x, y, bins=10):
    """Empirical mutual information between a continuous feature x and binary label y."""
    p_y = np.bincount(y) / len(y)
    H_y = entropy(p_y, base=2)
    # H(Y|X): bin x, compute H(Y|X=bin) for each bin, weighted average
    bin_idx = np.digitize(x, np.linspace(x.min(), x.max(), bins + 1)) - 1
    bin_idx = np.clip(bin_idx, 0, bins - 1)
    H_y_given_x = 0.0
    for b in range(bins):
        mask = (bin_idx == b)
        if mask.sum() == 0:
            continue
        y_b = y[mask]
        p_b = mask.mean()
        p_y_b = np.bincount(y_b, minlength=2) / len(y_b)
        H_y_given_x += p_b * entropy(p_y_b, base=2)
    return H_y - H_y_given_x

n = 2000
y = rng.binomial(1, 0.5, n)

# Three features: informative, weakly informative, noise
feature_informative = np.where(y == 1, rng.normal(2, 1, n), rng.normal(-2, 1, n))
feature_weak        = np.where(y == 1, rng.normal(0.5, 2, n), rng.normal(-0.5, 2, n))
feature_noise       = rng.normal(0, 1, n)

for name, feat in [('Informative', feature_informative), ('Weak', feature_weak), ('Noise', feature_noise)]:
    mi = mutual_information_discrete(feat, y)
    print(f'MI(feature={name}, label): {mi:.4f} bits')

## ✏️ Your turn

**Task A — Entropy of a die:** Compute the entropy of a fair 6-sided die (in bits). Then compute the entropy of a loaded die with $P = [0.5, 0.3, 0.1, 0.05, 0.03, 0.02]$. How many bits does the bias save?

**Task B — KL is not symmetric:** Compute $D_{KL}(P \| Q)$ and $D_{KL}(Q \| P)$ for $P = [0.9, 0.1]$ and $Q = [0.5, 0.5]$. Which direction produces the larger value, and why?

In [ ]:
# Task A
P_fair   = np.ones(6) / 6
P_loaded = np.array([0.5, 0.3, 0.1, 0.05, 0.03, 0.02])
# TODO(you): compute entropy of each, find the difference

# Task B
P_b = np.array([0.9, 0.1])
Q_b = np.array([0.5, 0.5])
# TODO(you): compute KL(P||Q) and KL(Q||P), explain which is larger and why

<details><summary>Solution</summary>

```python
# Task A
print(f'Fair die:   H = {entropy(P_fair):.4f} bits')    # log2(6) ≈ 2.585
print(f'Loaded die: H = {entropy(P_loaded):.4f} bits')  # < 2.585
print(f'Savings: {entropy(P_fair) - entropy(P_loaded):.4f} bits')

# Task B
P_b = np.array([0.9, 0.1]); Q_b = np.array([0.5, 0.5])
print(f'KL(P||Q) = {kl_divergence(P_b, Q_b):.4f} nats')  # ~0.51 — large: P concentrated, Q spread
print(f'KL(Q||P) = {kl_divergence(Q_b, P_b):.4f} nats')  # ~1.30 — larger: Q puts mass where P≈0
# KL(Q||P) is larger because Q(0)=0.5 but P(0)=0.1, and log(Q/P) at that point is log(5)
```
</details>